# ДЗ 2.1 — SCD2

Собираю историю профилей в Spark и сохраняю её в RustFS S3. Время — UTC. Для user_id использую BIGINT с проверкой диапазона uint32.

In [1]:
from scd2 import make_spark, read_events, build_table
from workflow import (prepare_source, SOURCE, TARGET, edge_checks,
                      verify_result, lookup_checks, save_s3_objects)

spark = make_spark()
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version)
print('Session timezone:', spark.conf.get('spark.sql.session.timeZone'))

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 16:33:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.3


Session timezone: UTC


## 1. Исходные данные

Генератором из задания создаю 5376 событий и загружаю CSV в S3. Число строк выбираю случайно в диапазоне 5000–7000; фиксирую seed для повторного запуска.

In [2]:
manifest, s3 = prepare_source()
events = read_events(spark, SOURCE)
assert events.count() == manifest['total_rows']
events.printSchema()
events.orderBy('event_time').show(10, truncate=False)

files=6/6 rows=5376/5376 rate=37,758 rows/s


Готово: 5376 строк в каталоге /work/data/event_log


{
  "format": "csv",
  "columns": [
    "event_time",
    "user_id",
    "user_fio",
    "user_email",
    "user_login"
  ],
  "total_rows": 5376,
  "rows_per_file": 1000,
  "file_count": 6,
  "user_count": 268,
  "user_id_start": 1,
  "change_probability": 0.01,
  "applied_profile_changes": 54,
  "seed": 42,
  "first_event_time": "2026-09-01T00:00:03Z",
  "last_event_time": "2026-09-01T03:01:23Z",
  "elapsed_seconds": 0.142
}
Uploaded source to s3a://hw-2-1/source/event_log/*.csv


root
 |-- event_time: timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)



+-------------------+-------+------------------+----------------------------------+-----------+
|event_time         |user_id|user_fio          |user_email                        |user_login |
+-------------------+-------+------------------+----------------------------------+-----------+
|2026-09-01 00:00:03|58     |Aiden Howard      |aiden.howard.58@mail.example      |ahoward58  |
|2026-09-01 00:00:04|72     |Ashley Williams   |ashley.williams.72@webmail.example|awilliams72|
|2026-09-01 00:00:05|217    |Alexander Baker   |alexander.baker.217@example.com   |abaker217  |
|2026-09-01 00:00:06|120    |Lucas Stewart     |lucas.stewart.120@example.com     |lstewart120|
|2026-09-01 00:00:09|102    |Taylor Wright     |taylor.wright.102@webmail.example |twright102 |
|2026-09-01 00:00:11|113    |Justin Scott      |justin.scott.113@inbox.example    |jscott113  |
|2026-09-01 00:00:12|82     |Joseph King       |joseph.king.82@inbox.example      |jking82    |
|2026-09-01 00:00:13|111    |Henry Clark

## 2. Сборка истории

Сравниваю профиль с предыдущим событием пользователя. Создаю версию только при изменении атрибутов. Через lead задаю valid_to; у последней версии оставляю NULL. Код сборки — в scd2.py.

In [3]:
first_count = build_table(spark, SOURCE, TARGET)
first_rows = spark.read.parquet(TARGET).collect()
print('First build rows:', first_count)
spark.read.parquet(TARGET).orderBy('user_id', 'valid_from').show(15, truncate=False)

First build rows: 322
+-------+-----------------+-----------------------------------+------------+-------------------+-------------------+----------+
|user_id|user_fio         |user_email                         |user_login  |valid_from         |valid_to           |is_current|
+-------+-----------------+-----------------------------------+------------+-------------------+-------------------+----------+
|1      |Evelyn Hughes    |evelyn.hughes.1@mail.example       |ehughes1    |2026-09-01 00:03:47|NULL               |true      |
|2      |Emma Davis       |emma.davis.2@mail.example          |edavis2     |2026-09-01 00:10:41|NULL               |true      |
|3      |Ashley Murphy    |ashley.murphy.3@mail.example       |amurphy3    |2026-09-01 00:04:04|NULL               |true      |
|4      |Christopher James|christopher.james.4@mail.example   |cjames4     |2026-09-01 00:12:15|NULL               |true      |
|5      |Austin Bryant    |austin.bryant.5@mail.example       |abryant5    |2026-0

## 3. Проверки

Повторно собираю таблицу и сравниваю результаты. Проверяю интервалы, одну текущую версию на пользователя и совпадение с расчётом на Python. Отдельно проверяю NULL, дубли и возврат к прежнему профилю.

In [4]:
second_count = build_table(spark, SOURCE, TARGET)
assert second_count == first_count
history, report = verify_result(spark, first_rows, manifest)
assert edge_checks(spark)
print(report)

SELECT count(*) AS scd2_rows
FROM users_scd2;
 
Result: 322


Edge checks passed: NULL, A→B→A, duplicates, uint32, timestamp conflicts
{'source_rows': 5376, 'users': 268, 'changes': 54, 'scd2_rows': 322, 'snapshot_sha256': '18e146eff547957ab6bd842ac68caf64d15e952eb4bfbf8f056733f6adf6da56', 'rerun_equal': True, 'independent_reference_equal': True, 'invariants_passed': True}


## 4. Количество строк

Выполняю запрос из queries/count.sql. В итоговой таблице 322 строки.

In [5]:
from pathlib import Path
count_sql = Path('queries/count.sql').read_text(encoding='utf-8')
print(count_sql)
spark.sql(count_sql).show()

SELECT count(*) AS scd2_rows
FROM users_scd2;

+---------+
|scd2_rows|
+---------+
|      322|
+---------+



## 5. Логин на заданный момент

Передаю user_id и as_of параметрами в запрос из queries/login_at.sql. Проверяю границы интервалов: до первого события строк нет, в момент изменения выбирается новая версия.

In [6]:
lookup_sql = Path('queries/login_at.sql').read_text(encoding='utf-8')
print(lookup_sql)
lookups = lookup_checks(spark, history)

-- Parameters are bound by SparkSession.sql(args=...), never interpolated.
-- Session timezone: UTC. Intervals are [valid_from, valid_to).
SELECT user_login
FROM users_scd2
WHERE user_id = :user_id
  AND valid_from <= CAST(:as_of AS TIMESTAMP)
  AND (valid_to IS NULL OR CAST(:as_of AS TIMESTAMP) < valid_to);



before_first {'user_id': 24, 'as_of': datetime.datetime(2026, 9, 1, 0, 12, 13, 999999)} => []
at_first {'user_id': 24, 'as_of': datetime.datetime(2026, 9, 1, 0, 12, 14)} => ['dmitchell24']
before_change {'user_id': 24, 'as_of': datetime.datetime(2026, 9, 1, 1, 20, 28, 999999)} => ['dmitchell24']


at_change {'user_id': 24, 'as_of': datetime.datetime(2026, 9, 1, 1, 20, 29)} => ['twalker24r2']
current {'user_id': 24, 'as_of': datetime.datetime(2026, 9, 2, 1, 20, 29)} => ['twalker24r2']


## 6. Сохранение результатов

Сохраняю результаты проверок и список объектов S3. Копия истории для просмотра — results/users_scd2.csv.

In [7]:
save_s3_objects(s3)
history.unpersist()
spark.stop()

Список объектов S3 сохранён в results/s3_objects.json
